# धडा 08 - मल्टी-एजंट डिझाईन पॅटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मल्टी-एजंट सिस्टम का?

वास्तविक जगातील कामांमध्ये, जसे की सफर नियोजनामध्ये, अनेक प्रकारच्या तज्ञतेची गरज असते — लॉजिस्टिक्स, स्थानिक ज्ञान, बजेटिंग, आणि बरेच काही. एकच एजंट सर्व काही हाताळण्याचा प्रयत्न करणारा जलदच अव्यवस्थित होतो.

मल्टी-एजंट सिस्टम्स हे **विशेषीकरण** द्वारे याचे समाधान करतात: प्रत्येक एजंट एका विशिष्ट तज्ञतेवर लक्ष केंद्रित करतो, ज्यामुळे सामान्य तज्ञापेक्षा जास्त दर्जेदार परिणाम येतात. ते **स्केलेबिलिटी** सुध्दा सुधारतात — तुम्ही नवीन एजंट्स (उदा., फ्लाइट विशेषज्ञ, रेस्टॉरंट समीक्षक) आधीच्या वर्कफ्लोला पुन्हा लिहिण्याशिवाय जोडू शकता. एजंट्स एकत्रितपणे एका रचलेल्या पाईपलाइनद्वारे एकमेकांकडून संदर्भ Passing करतात.


## विशेष एजंट तयार करणे


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## क्रमवारीकृत कार्यप्रवाह तयार करणे

`WorkflowBuilder` आपल्याला एजंट्सना निर्देशित ग्राफमध्ये वायर करण्याची परवानगी देते. येथे आपण एक सोपी दोन-चरणांची पाईपलाईन तयार करतो: **TravelPlanner** प्रवासाचे आराखडा तयार करतो, नंतर **TravelConcierge** त्याचे पुनरावलोकन करतो आणि त्यात सुधारणा करतो.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ्लोमध्ये आणखी एजंट जोडणे

मल्टी-एजंट पॅटर्नचा एक मोठा फायदा म्हणजे त्याचा विस्तार करणे किती सोपे आहे. खाली आपण एक **BudgetReviewer** एजंट जोडतो जो प्रवाशाच्या बजेटच्या तुलनेत योजनेची तपासणी करतो, अशा गोष्टींना चिन्हांकित करतो ज्या खर्च मर्यादेपलीकडे जाऊ शकतात, आणि पैसे वाचवण्याच्या पर्यायांचा प्रस्ताव करतो. वर्कफ्लो आता सलग तीन एजंट चालवतो:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

या धड्यात तुम्ही काय शिकले:

1. **विशेषीकृत एजंट तयार करणे** — ज्यामध्ये प्रत्येकाचा एक लक्ष केंद्रीत भूमिका असते (योजना, कन्सीएर्ज, बजेट पुनरावलोकन).
2. **`WorkflowBuilder` आणि `add_edge` वापरून एजंट्सना अनुक्रमिक कार्यप्रवाहात जोडणे.**
3. **मल्टि-एजंट पाइपलाइनमधून आउटपुट प्रवाहित करणे, कोणता एजंट बोलत आहे हे ट्रॅक करणे.**
4. **कार्यप्रवाह वाढवणे** — नवीन एजंट्स साखळीत जोडणे, आधीच्या एजंट्सना बदल न करता.

मल्टि-एजंट डिझाईन पॅटर्न प्रत्येक एजंटला सोपे ठेवतो, पण एकल एजंट पेक्षा अधिक समृद्ध आणि अधिक सखोल पुनरावलोकित परिणाम तयार करतो.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरीही कृत्रिम अनुवादांमध्ये त्रुटी किंवा अचूकतेच्या अभाव असू शकतो याची कृपया जाणीव ठेवा. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाचा वापर केल्यामुळे होणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थनिर्देशांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
